In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
catalog_name='ecommerce'

In [0]:
df_silver= spark.table(f"{catalog_name}.silver.slv_order_items")
display(df_silver.limit(5))

In [0]:
df_gold = df_silver.withColumn("gross_amount",f.col("quantity")*f.col("unit_price"))\
    .withColumn("discount_amount",f.ceil(f.col("gross_amount")*f.col("discount_pct")/100))\
    .withColumn("sale_amount",f.col("gross_amount")-f.col("discount_amount")+f.col("tax_amount"))\
    .withColumn("date_id",f.date_format(f.col("dt"),"yyyyMMdd").cast("int"))\
    .withColumn("coupon_flag",f.when(f.col("coupon_code").isNotNull(),f.lit(1)).otherwise(f.lit(0)))

display(df_gold.limit(5))

In [0]:
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ["currency", "inr_rate"])
rates_df.show()

In [0]:
df_gold = (
    df_gold
    .join(
        rates_df,
        rates_df.currency == f.upper(f.trim(f.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sale_amount_inr", f.ceil(f.col("sale_amount") * f.col("inr_rate")))
)
display(df_gold.limit(5))

In [0]:
orders_gold_df = df_gold.select(
    f.col("date_id"),
    f.col("dt").alias("transaction_date"),
    f.col("order_ts").alias("transaction_ts"),
    f.col("order_id").alias("transaction_id"),
    f.col("customer_id"),
    f.col("item_seq").alias("seq_no"),
    f.col("product_id"),
    f.col("channel"),
    f.col("coupon_code"),
    f.col("coupon_flag"),
    f.col("unit_price_currency"),
    f.col("quantity"),
    f.col("unit_price"),
    f.col("gross_amount"),
    f.col("discount_pct").alias("discount_percent"),
    f.col("discount_amount"),
    f.col("tax_amount"),
    f.col("sale_amount").alias("net_amount"),
    f.col("sale_amount_inr").alias("net_amount_inr")
)

In [0]:
orders_gold_df.write.format("delta")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")

In [0]:
%sql

select count(*) from ecommerce.gold.gld_fact_order_items 